In [0]:
import os
import sys

sys.path.append(os.path.abspath("../../src"))

from pipeline.silver.cdc import (
    parse_envelope,
    convert_types,
    add_versions,
    latest_per_key,
)
from pipeline.silver.registry import TABLES

In [0]:
dbutils.widgets.text("table", "")
TABLE = dbutils.widgets.get("table")

In [0]:
if TABLE not in TABLES:
    raise ValueError(f"table must be one of {list(TABLES)}, got '{TABLE}'")

CONFIG = TABLES[TABLE]
if not CONFIG.get("key") or not CONFIG.get("schema"):
    raise ValueError(f"registry entry for '{TABLE}' is incomplete")
if CONFIG.get("scd") not in (1, 2):
    raise ValueError(f"registry entry for '{TABLE}' needs scd = 1 or 2")

SOURCE = f"motorsport.bronze.cdc_{TABLE}"
TARGET = f"motorsport.silver.{TABLE}"

In [0]:
bronze = spark.table(SOURCE)

parsed = parse_envelope(bronze, CONFIG)
typed = convert_types(parsed, CONFIG)

final = (
    add_versions(typed, CONFIG["key"])
    if CONFIG["scd"] == 2
    else latest_per_key(typed, CONFIG["key"])
)


In [0]:
final.write.mode("overwrite").saveAsTable(TARGET)

In [0]:
%sql
SHOW TABLES IN motorsport.silver;